In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

%matplotlib inline

Analyze the assignment

In [ ]:
# Test case
res = 90
procs = 144
case = f"c{res}_p{procs}"

reassign_types = [
    "greedy",
    "nearest_greedy",
    "bilinear_greedy",
    "bicubic_greedy",
]
reassgins = {}

In [ ]:
# Load the CSV files
og_assign = pd.read_csv(f"test/og_assignments/{case}.csv", index_col=0)
for sim_type in reassign_types:
    file_path = f"test/{sim_type}/{case}/assignment.csv"
    reassgins[sim_type] = pd.read_csv(file_path, index_col=0)

# Trim the original dataframe to match the upscaled intervals
upscaled_df = reassgins["bilinear_greedy"]
original_df = reassgins["greedy"]
original_df = original_df[upscaled_df.columns]

# Rename the columns to match the upscaled intervals
nintervals = len(upscaled_df.columns)
intervals = range(nintervals)
upscaled_df.columns = intervals
original_df.columns = intervals

In [ ]:
# Track the target processor the original and reassignment decide to assign for each processor at each interval
# Also track how many times it is assigned to that processor
# Assuming there is up to one target processor per processor at each interval
def count_assignments(
    assign: pd.DataFrame, og_assign: pd.DataFrame, procs: int
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Count the number of assignments for each processor at each interval.
    """
    samples, intervals = assign.shape
    counts = np.zeros((procs, intervals), dtype=int)
    targets = -np.ones((procs, intervals), dtype=int)

    # Expand original assignments to match shape of assign
    og_expanded = np.tile(og_assign.values, (1, intervals))

    # Mask where assignments differ
    mask = assign.values != og_expanded

    # Get processor indices and interval indices where reassignment occurred
    reassigned_proc = assign.values[mask]
    original_proc = og_expanded[mask]
    interval_idx = np.tile(np.arange(intervals), samples)[mask.ravel()]

    # Count assignments and set targets
    for proc, orig, idx in zip(reassigned_proc, original_proc, interval_idx):
        counts[proc, idx] += 1
        targets[proc, idx] = orig

    return pd.DataFrame(
        counts, index=range(procs), columns=assign.columns
    ), pd.DataFrame(targets, index=range(procs), columns=assign.columns)

In [ ]:
# Count it for greedy heuristic on both the original and upscaled workloads
og_counts, og_targets = count_assignments(original_df, og_assign, procs)

# Print the first interval of the counts and targets
print("Original workload counts:", og_counts.iloc[:, 0].values)
print("Original workload targets:", og_targets.iloc[:, 0].values)

In [ ]:
upscaled_counts, upscaled_targets = count_assignments(upscaled_df, og_assign, procs)

print("Upscaled workload counts:", upscaled_counts.iloc[:, 0].values)
print("Upscaled workload targets:", upscaled_targets.iloc[:, 0].values)

In [ ]:
# Tally for each interval, how many times does the upscaled workload assigns to a different processor than the original workload
target_diff = upscaled_targets != og_targets
target_diff_count = target_diff.sum(axis=0)

print("Target difference counts:", target_diff_count.values)


Compare the workload
Hypothesis: the highest workload and lowest workload processors are predicted correctly.

In [ ]:
# Read the workload data
workload_df = pd.read_csv(f"test/workloads/c{res}.csv", index_col=0, header=0)
bilinear_workload_df = pd.read_csv(f"test/workloads/bilinear_c24_to_c{res}.csv", index_col=0, header=0)

# Trim the original workload to match the upscaled intervals
workload_df = workload_df[bilinear_workload_df.columns]

In [ ]:
# Compute the workload for each processor at each interval using the original assignments
def compute_workload(
    workload_df: pd.DataFrame, og_assign: pd.DataFrame, procs: int
) -> pd.DataFrame:
    """
    Compute the workload for each processor at each interval.
    """
    _, intervals = workload_df.shape
    workload = np.zeros((procs, intervals), dtype=float)
    assigned_procs = og_assign.values.flatten()

    for interval in range(intervals):
        np.add.at(
            workload[:, interval], assigned_procs, workload_df.values[:, interval]
        )

    return pd.DataFrame(workload, index=range(procs), columns=workload_df.columns)

In [ ]:
# Compute the workload for the original workload df
og_workload = compute_workload(workload_df, og_assign, procs)

og_max_indices = og_workload.idxmax(axis=0)
rows = og_max_indices.values
cols = np.arange(len(og_workload.columns))
og_max_values = og_workload.values[rows, cols]

print("Original max workload processors:", og_max_indices)

In [ ]:
# Compute the workload for the bilinear workload df
bilinear_workload = compute_workload(bilinear_workload_df, og_assign, procs)

bilinear_max_indices = bilinear_workload.idxmax(axis=0)
rows = bilinear_max_indices.values
cols = np.arange(len(bilinear_workload.columns))
bilinear_max_values = bilinear_workload.values[rows, cols]

print("Bilinear max workload processors:", bilinear_max_indices)

In [ ]:
# Check how many times the upscaled workload predicts the max workload processor correctly
correct_predictions = bilinear_max_indices == og_max_indices
correct_predictions_count = correct_predictions.sum()
print("Correct predictions count:", correct_predictions_count)

# For each of those correct predictions, check how far off the predicted workload is from the actual workload
ratios = bilinear_max_values[correct_predictions] / og_max_values[correct_predictions]

# Compute summary statistics for the ratios
median_ratio = np.nanmedian(ratios)
geo_mean_ratio = np.exp(np.nanmean(np.log(ratios[ratios > 0])))
within_1_percent = np.mean((ratios >= 0.99) & (ratios <= 1.01)) * 100
mape = np.mean(np.abs(1 - ratios)) * 100

# Repeat the computation but including all predictions
ratios_all = bilinear_max_values / og_max_values

# Compute summary statistics for the ratios
median_ratio_all = np.nanmedian(ratios_all)
geo_mean_ratio_all = np.exp(np.nanmean(np.log(ratios_all[ratios_all > 0])))
within_1_percent_all = np.mean((ratios_all >= 0.99) & (ratios_all <= 1.01)) * 100
mape_all = np.mean(np.abs(1 - ratios_all)) * 100

# Create summary table
summary_table = pd.DataFrame(
    {
        "Median Ratio": [median_ratio, median_ratio_all],
        "Geometric Mean Ratio": [geo_mean_ratio, geo_mean_ratio_all],
        "% Within 1%": [within_1_percent, within_1_percent_all],
        "MAPE (%)": [mape, mape_all],
    },
    index=["Correct", "All"],
)

print(summary_table)

In [ ]:
# Absolute difference
abs_diff_correct = np.abs(
    bilinear_max_values[correct_predictions] - og_max_values[correct_predictions]
)
abs_diff_all = np.abs(bilinear_max_values - og_max_values)


abs_diff_table = pd.DataFrame(
    {
        "mean": [np.mean(abs_diff_correct), np.mean(abs_diff_all)],
        "std": [np.std(abs_diff_correct), np.std(abs_diff_all)],
        "min": [np.min(abs_diff_correct), np.min(abs_diff_all)],
        "max": [np.max(abs_diff_correct), np.max(abs_diff_all)],
        "median": [np.median(abs_diff_correct), np.median(abs_diff_all)],
    },
    index=["Correct", "All"],
)
print(abs_diff_table)

In [ ]:
# Absolute ratio deviation
ratios_correct = (
    bilinear_max_values[correct_predictions] / og_max_values[correct_predictions]
)
ratios_all = bilinear_max_values / og_max_values
abs_ratio_dev_correct = np.abs(ratios_correct - 1)
abs_ratio_dev_all = np.abs(ratios_all - 1)

abs_ratio_table = pd.DataFrame(
    {
        "mean": [
            abs_ratio_dev_correct.mean(),
            abs_ratio_dev_all.mean(),
        ],
        "std": [
            np.std(abs_ratio_dev_correct),
            np.std(abs_ratio_dev_all),
        ],
        "min": [
            np.min(abs_ratio_dev_correct),
            np.min(abs_ratio_dev_all),
        ],
        "median": [
            np.median(abs_ratio_dev_correct),
            np.median(abs_ratio_dev_all),
        ],
        "max": [
            np.max(abs_ratio_dev_correct),
            np.max(abs_ratio_dev_all),
        ],
    },
    index=["Correct", "All"],
)
print(abs_ratio_table)

In [ ]:
# Now rather than just comparing the max workload per processor, compare the entire workload
# Compute the absolute difference between the bilinear and original workloads, and summarize the results
abs_diff_workload = np.abs(bilinear_workload.values - og_workload.values)
abs_diff_workload_df = pd.DataFrame(
    abs_diff_workload, index=range(procs), columns=workload_df.columns
)
abs_diff_workload_summary = abs_diff_workload_df.describe().T
# Drop the count, 25%, and 75% columns, and rename the 50% column to median
abs_diff_workload_summary.drop(columns=["count", "25%", "75%"], inplace=True)
abs_diff_workload_summary.rename(
    columns={
        "50%": "median",
    },
    inplace=True,
)
print(abs_diff_workload_summary)

In [ ]:
# Repeat the same for the ratios
ratios_workload = bilinear_workload.values / og_workload.values
ratios_workload_df = pd.DataFrame(
    ratios_workload, index=range(procs), columns=workload_df.columns
)
ratios_workload_summary = ratios_workload_df.describe().T
# Drop the count, 25%, and 75% columns, and rename the 50% column to median
ratios_workload_summary.drop(columns=["count", "25%", "75%"], inplace=True)
ratios_workload_summary.rename(
    columns={
        "50%": "median",
    },
    inplace=True,
)
print(ratios_workload_summary)

In [ ]:
# And absolute ratio deviation
abs_ratio_dev_workload = np.abs(ratios_workload - 1)
abs_ratio_dev_workload_df = pd.DataFrame(
    abs_ratio_dev_workload, index=range(procs), columns=workload_df.columns
)
abs_ratio_dev_workload_summary = abs_ratio_dev_workload_df.describe().T
# Drop the count, 25%, and 75% columns, and rename the 50% column to median
abs_ratio_dev_workload_summary.drop(columns=["count", "25%", "75%"], inplace=True)
abs_ratio_dev_workload_summary.rename(
    columns={
        "50%": "median",
    },
    inplace=True,
)
print(abs_ratio_dev_workload_summary)

In [ ]:
from scipy.stats import spearmanr

nearest_workload_df = pd.read_csv(
    f"test/workloads/nearest_c24_to_c{res}.csv", index_col=0, header=0
)
bilinear_workload_df = pd.read_csv(
    f"test/workloads/bilinear_c24_to_c{res}.csv", index_col=0, header=0
)
bicubic_workload_df = pd.read_csv(
    f"test/workloads/bicubic_c24_to_c{res}.csv", index_col=0, header=0
)

methods = {
    "nearest": compute_workload(nearest_workload_df, og_assign, procs),
    "bilinear": compute_workload(bilinear_workload_df, og_assign, procs),
    "bicubic": compute_workload(bicubic_workload_df, og_assign, procs),
}

summary_table = {}

for name, pred_workload in methods.items():
    correlations = [
        spearmanr(og_workload.iloc[i].values, pred_workload.iloc[i].values)[0]
        for i in range(procs)
    ]
    stats = pd.Series(correlations).describe()
    summary_table[name] = {
        "mean": stats["mean"],
        "std": stats["std"],
        "min": stats["min"],
        "median": stats["50%"],
        "max": stats["max"],
    }

# Create and display the final summary table
correlation_summary_df = pd.DataFrame(summary_table).T[
    ["mean", "std", "min", "median", "max"]
]
print(correlation_summary_df)

In [ ]:
# Compute the spearman correlation for each interval
def compute_spearman_correlation(
    og_workload: pd.DataFrame, pred_workload: pd.DataFrame
) -> pd.Series:
    """
    Compute the spearman correlation for each interval.
    """
    correlations = [
        spearmanr(og_workload.iloc[:, i].values, pred_workload.iloc[:, i].values)[0]
        for i in range(og_workload.shape[1])
    ]
    return pd.Series(correlations)

# Compute the spearman correlation
nearest_correlation = compute_spearman_correlation(workload_df, nearest_workload_df)
bilinear_correlation = compute_spearman_correlation(workload_df, bilinear_workload_df)
bicubic_correlation = compute_spearman_correlation(workload_df, bicubic_workload_df)

# Summary statistics
nearest_correlation_summary = nearest_correlation.describe()
nearest_correlation_summary.drop(index=["count", "25%", "75%"], inplace=True)
nearest_correlation_summary.rename(
    index={
        "50%": "median",
    },
    inplace=True,
)
bilinear_correlation_summary = bilinear_correlation.describe()
bilinear_correlation_summary.drop(index=["count", "25%", "75%"], inplace=True)
bilinear_correlation_summary.rename(
    index={
        "50%": "median",
    },
    inplace=True,
)
bicubic_correlation_summary = bicubic_correlation.describe()
bicubic_correlation_summary.drop(index=["count", "25%", "75%"], inplace=True)
bicubic_correlation_summary.rename(
    index={
        "50%": "median",
    },
    inplace=True,
)

# Combine all summaries into a single table
correlation_table = pd.DataFrame(
    {
        "nearest": nearest_correlation_summary,
        "bilinear": bilinear_correlation_summary,
        "bicubic": bicubic_correlation_summary,
    }
)

# Print the table with comma as separator
print(correlation_table.to_string(index=True, header=True, float_format="%.3f"))

In [ ]:
import matplotlib.pyplot as plt

interval = 0  # start with a representative interval
x = workload_df.iloc[:, interval].array
y = bilinear_workload_df.iloc[:, interval].array

plt.figure(figsize=(6, 6))
plt.scatter(x, y, alpha=0.5, s=5)
plt.plot([x.min(), x.max()], [x.min(), x.max()], color="red", linestyle="--")
plt.xlabel("Ground Truth Workload")
plt.ylabel("Upscaled (Bilinear)")
plt.title(f"Upscaled vs. Ground Truth (Interval {interval})")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Repeat it but using data from all intervals
x = workload_df.values.flatten()
y = bilinear_workload_df.values.flatten()
plt.figure(figsize=(6, 6))
plt.scatter(x, y, alpha=0.5, s=5)
plt.plot([x.min(), x.max()], [x.min(), x.max()], color="red", linestyle="--")
plt.xlabel("Ground Truth Workload")
plt.ylabel("Upscaled (Bilinear)")
plt.title(f"C{res} - Upscaled vs. Ground Truth")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# The plot shows a bunch of zeros for upscaled, check if that's the case.
print(bilinear_workload_df[bilinear_workload_df == 0].count().sum())

Compare the simulated workspan

In [ ]:
# Read all the simulation data
simulation_types = ["original", "greedy", "nearest_greedy", "bilinear_greedy", "bicubic_greedy"]
simulations = {}

for sim_type in simulation_types:
    if sim_type == "original":
        file_path = f"test/og_assignments/{case}_simulation.csv"
    else:
        file_path = f"test/{sim_type}/{case}/simulation.csv"
    simulations[sim_type] = pd.read_csv(file_path)

In [ ]:
# Generate an animation showing the workload distribution for all the simulation types
fig, ax = plt.subplots(figsize=(160, 6))
bar_width = 0.15
bars = []

# Exclude 'Interval' column and the last 4 Max,Mean,SD,CV columns
ylim = max(simulations[simulation_types[i]].iloc[:, 1:-4].max().max() * 1.1 for i in range(len(simulation_types)))
xlabels = simulations[simulation_types[0]].columns[1:-4]
x = np.arange(len(xlabels))

def init():
    ax.clear()
    ax.margins(x=0)
    ax.set_ylim(0, ylim)
    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, rotation=90)
    ax.set_ylabel("Workload")
    ax.set_title("Workload Distribution per Processor")
    fig.tight_layout()
    return bars

def update(frame):
    ax.clear()
    ax.margins(x=0)
    for i, sim_type in enumerate(simulation_types):
        sim_data = simulations[sim_type]
        workloads = sim_data.iloc[frame, 1:-4]
        bar = ax.bar(x + i * bar_width, workloads, bar_width, label=sim_type)
        bars.append(bar)
    # Plot a dotted reference line for the mean workload
    mean = simulations[simulation_types[0]].iloc[frame, -3]
    ax.axhline(mean, color="red", linestyle="--", label="Mean Workload")

    ax.set_ylim(0, ylim)
    ax.set_xticks(x + bar_width * (len(simulation_types)-1)/2)
    ax.set_xticklabels(xlabels, rotation=90)
    ax.set_ylabel("Workload")
    ax.set_title(f"Workload Distribution per Processor (Interval {frame})")
    ax.legend(
        loc="upper right",
        title="Simulation Type",
        fontsize="small",
    )
    fig.tight_layout()
    return bars

ani = FuncAnimation(
    fig,
    update,
    frames=range(len(simulations[simulation_types[0]])),
    init_func=init,
    blit=False,
)

ani.save(f"{case}_anim.gif", writer="imagemagick")